# Notebook 1: Dataset and Pipeline Walkthrough

This notebook explains, step by step, how the repository loads the dataset, builds translation chains, and prepares experiment conditions.

It is meant as a readable companion to the code in `src/translation_chains/`.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from translation_chains.datasets import load_dataset_from_config
from translation_chains.adapters import make_translation_adapter, make_model_adapter

config_path = ROOT / 'configs' / 'runtime.experiment.json'
config = json.loads(config_path.read_text(encoding='utf-8'))
config

## Step 1: Load the dataset

The default runtime config currently uses a local JSONL sample dataset. If you switch to `configs/runtime.hf.example.json`, the same loader will fetch data from Hugging Face instead.

In [ ]:
dataset = load_dataset_from_config(config_path, config)
len(dataset), dataset[0]

## Step 2: Instantiate adapters

The repository ships with deterministic local adapters so you can test the pipeline without external APIs.

In [ ]:
translator = make_translation_adapter(config['translation_engine'])
models = [make_model_adapter(name) for name in config['models']]
translator.name, [model.name for model in models]

## Step 3: Inspect a translation chain

Each experiment condition starts from the original instruction and applies one translation step at a time. The code evaluates after every step.

In [ ]:
record = dataset[0]
path = config['paths'][0]
current = record.original_prompt
steps = []
for source_lang, target_lang in zip(path, path[1:]):
    current = translator.translate(current, source_lang, target_lang)
    steps.append((source_lang, target_lang, current))
steps

## Step 4: Inspect model behavior on a translated prompt

This shows the prompt the model sees after drift and the response produced by one of the local model adapters.

In [ ]:
prompt_after_chain = steps[-1][2]
fragile = make_model_adapter('fragile')
print(prompt_after_chain)
print('---')
print(fragile.generate(record, prompt_after_chain))